# Example of model comparison with Harmonic: TOI-544

In this notebook, we demonstrate how to use the [harmonic](https://astro-informatics.github.io/harmonic/) package to perform Bayesian model comparison for exoplanet radial velocity models. We compare a 1-planet model against a 2-planet model for the TOI-544 system. This system was first characterised in RV by [Osborne et al. (2024)](https://doi.org/10.1093/mnras/stad3837), who knew TOI-544~b was transiting, but suspected the presence of an outer non-transiting planet TOI-544~c.

In Bayesian inference, the Bayesian evidence (marginal likelihood) $\mathcal{Z} = \int \mathcal{L}(\theta)\, \pi(\theta)\, d\theta$ quantifies how well a model explains the data, integrating over the full prior volume. Unlike information criteria (AIC, BIC), it naturally penalises unnecessary model complexity through the prior -- rather than just through the likelihood and the dimensionality -- this is the Bayesian Occam's razor. In RV we tend to have quite informative priors (e.g. tight priors on $P$ from transits, or on quasiperiodic GP kernel hyperparameters through photometric studies), so it seems a waste to not use that prior information in our model comparisons by doing AIC or BIC instead.

The problem though is that computing $\mathcal{Z}$ directly is computationally intractable for high-dimensional problems. In MCMC, we only sample the likelihood and the prior functions, not the evidence (giving us the *un-normalised* posterior, which is fine for parameter inference, but need it to be normalised for model comparison). Therefore, we will use the `harmonic` package implementation of the Learned Harmonic Mean Estimator (LHME) which uses normalizing flows to estimate $\mathcal{Z}$ from our existing posterior MCMC samples. See [McEwen et al. (2021)](https://doi.org/10.48550/arXiv.2111.12720) for the theoretical background, but the quick sell is that this is basically free - we already have the MCMC samples from the fitting/parameter inference anyway! We can use any sampling technique we like (such as `ravest`) and get evidences in five minutes.

This tutorial assumes familiarity with fitting RV models in `ravest` so we'll skip over the fine details -- see the [K2-24 case-study](K2-24.ipynb) and [GP fitting tutorial](example_GP.ipynb) for more detailed walkthroughs.

In [ ]:
from pathlib import Path

import harmonic as hm
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ravest.fit import GPFitter
from ravest.gp import GPKernel
from ravest.param import Parameter, Parameterisation
from ravest.prior import HalfNormal, Normal, Uniform

jax.config.update("jax_enable_x64", True)

## Data

TOI-544 is an M dwarf observed with HARPS and HARPS-N. The data below have already been prepared: radial velocities are in m/s and times are in BTJD (BJD - 2457000). This data is based on RV data originally from [Osborne et al. (2024) | MNRAS 527, 11138–11157](https://doi.org/10.1093/mnras/stad3837).

In [ ]:
fpath = Path.cwd() / "example_data" / "TOI-544.csv"
data = pd.read_csv(fpath)
print("Instruments in the dataset:", data["Instrument"].unique())
data.head()

In [ ]:
plt.figure(figsize=(15, 3.5))
plt.title("TOI-544 radial velocity data")
plt.ylabel("Radial Velocity [m/s]")
plt.xlabel("BTJD (days)")
for inst in data["Instrument"].unique():
    mask = data["Instrument"] == inst
    plt.errorbar(
        data.loc[mask, "BTJD"],
        data.loc[mask, "RV"],
        yerr=data.loc[mask, "e_RV"],
        marker=".",
        label=inst,
        linestyle="None",
    )
plt.legend()
plt.show()

## 1-Planet Model

We first fit a model with only planet b ($P = 1.5484$ d) on a circular orbit, using a Quasiperiodic GP to model stellar activity. The GP hyperparameters for the stellar rotation period and evolution timescale are constrained by values reported in Osborne et al. (2024).

We keep the fitting details compact here --- see the [GP fitting tutorial](example_GP.ipynb) for a thorough walkthrough of the GP fitting workflow.

In [ ]:
time = data["BTJD"].to_numpy(dtype=float)
rv = data["RV"].to_numpy(dtype=float)
e_rv = data["e_RV"].to_numpy(dtype=float)
instruments = data["Instrument"].to_numpy(dtype=str)
t0 = np.median(time)

unique_instruments = sorted(np.unique(instruments).tolist())
print(f"Instruments: {unique_instruments}")

In [ ]:
gpkernel_1pl = GPKernel(kernel_type="Quasiperiodic")
fitter_1pl = GPFitter(
    planet_letters=["b"],
    parameterisation=Parameterisation("P K secosw sesinw Tc"),
    gp_kernel=gpkernel_1pl,
)
fitter_1pl.add_data(time=time, vel=rv, velerr=e_rv, instrument=instruments, t0=t0)

In [ ]:
# Planet b: circular orbit (secosw, sesinw fixed to 0)
tc_b = 2459199.0314 - 2457000  # BTJD

# Per-instrument gamma bounds from the data range
rv_range = np.max(rv) - np.min(rv)
g_lower = np.min(rv) - 0.5 * rv_range
g_upper = np.max(rv) + 0.5 * rv_range

params_1pl = {
    "P_b": Parameter(1.5484, "d", fixed=False),
    "K_b": Parameter(10.0, "m/s", fixed=False),
    "secosw_b": Parameter(0.0, "", fixed=True),
    "sesinw_b": Parameter(0.0, "", fixed=True),
    "Tc_b": Parameter(tc_b, "d", fixed=False),
    "gd": Parameter(0.0, "m/s/d", fixed=True),
    "gdd": Parameter(0.0, "m/s/d^2", fixed=True),
}
for inst in unique_instruments:
    mask = instruments == inst
    params_1pl[f"g_{inst}"] = Parameter(np.median(rv[mask]), "m/s", fixed=False)
    params_1pl[f"jit_{inst}"] = Parameter(0.0, "m/s", fixed=False)

priors_1pl = {
    "P_b": Normal(1.5484, 0.000002),
    "K_b": Uniform(0.0, 50.0),
    "Tc_b": Normal(tc_b, 0.0007),
}
for inst in unique_instruments:
    priors_1pl[f"g_{inst}"] = Uniform(g_lower, g_upper)
    priors_1pl[f"jit_{inst}"] = Uniform(0, 30)

hyperparams_1pl = {
    "gp_amp": Parameter(1.0, "", fixed=False),
    "gp_lambda_e": Parameter(112, "d", fixed=False),
    "gp_lambda_p": Parameter(0.519, "", fixed=False),
    "gp_period": Parameter(19.343, "d", fixed=False),
}

hyperpriors_1pl = {
    "gp_amp": Uniform(0, rv_range),
    "gp_lambda_e": Uniform(6, 112 + (3 * 28)),
    "gp_lambda_p": Uniform(0.1, 0.519 + (4 * 0.091)),
    "gp_period": Uniform(19.343 - 2, 19.343 + 2),
}

fitter_1pl.params = params_1pl
fitter_1pl.priors = priors_1pl
fitter_1pl.hyperparams = hyperparams_1pl
fitter_1pl.hyperpriors = hyperpriors_1pl

print(f"Free parameters: {list(fitter_1pl.free_params_dict.keys())}")
print(f"Free hyperparameters: {list(fitter_1pl.free_hyperparams_names)}")
print(f"Total dimensions: {fitter_1pl.ndim}")

### Fit the 1-planet model

For demonstration purposes, we use reduced MCMC settings here to make this notebook run quickly. For real runs you should use way more - perhaps at least 200 walkers!

In [ ]:
map_1pl = fitter_1pl.find_map_estimate(method="Powell")
print(f"MAP log-probability: {-map_1pl.fun:.2f}")

In [ ]:
nwalkers_1pl = 32  # Needs to be at least 2x ndims, but you should use hundreds! 
init_1pl = fitter_1pl.generate_initial_walker_positions_random(nwalkers=nwalkers_1pl)

fitter_1pl.run_mcmc(
    initial_positions=init_1pl,
    nwalkers=nwalkers_1pl,
    max_steps=50_000,
    check_convergence=True,
    convergence_check_interval=1000,
    convergence_check_start=15_000,
    progress=True,
    multiprocessing=True,
)
print("MCMC completed")

In [ ]:
total_steps_1pl, nwalkers_1pl, ndim_1pl = fitter_1pl.sampler.iteration, fitter_1pl.sampler.nwalkers, fitter_1pl.sampler.ndim


samples_per_walker = 1000  # we want to keep the final 1000 samples from each walker, after thinning by 10
thin_rate = 10
discard_start_1pl = total_steps_1pl - (samples_per_walker * thin_rate)

print(f"nsteps: {total_steps_1pl}, nwalkers: {nwalkers_1pl}, ndim: {ndim_1pl}")
print(f"Discarding first {discard_start_1pl} steps, then thinning by {thin_rate} to get {samples_per_walker} samples per walker.")
print(f"Total samples after discarding and thinning: {(total_steps_1pl - discard_start_1pl) // thin_rate * nwalkers_1pl}")

In [ ]:
samples_1pl = fitter_1pl.get_samples_np(
    discard_start=discard_start_1pl, thin=thin_rate, flat=False
)
lnprob_1pl = fitter_1pl.get_sampler_lnprob(
    discard_start=discard_start_1pl, thin=thin_rate, flat=False
)
print(f"Samples shape: {samples_1pl.shape}  (nsteps, nwalkers, ndim)")
print(f"Lnprob shape:  {lnprob_1pl.shape}  (nsteps, nwalkers)")

Check the corners to make sure the posteriors look converged and well-behaved

In [ ]:
fitter_1pl.plot_corner(discard_start=discard_start_1pl, thin=thin_rate, plot_datapoints=True)

## 2-Planet Model

Now we fit a model with planet b (as before) plus a candidate planet c at $P \sim 50.5$ d with free eccentricity. We use a `HalfNormal(0.32)` eccentricity prior based on the Van Eylen et al. (2019) single-transiting planet distribution.

In [ ]:
gpkernel_2pl = GPKernel(kernel_type="Quasiperiodic")
fitter_2pl = GPFitter(
    planet_letters=["b", "c"],
    parameterisation=Parameterisation("P K secosw sesinw Tc"),
    gp_kernel=gpkernel_2pl,
)
fitter_2pl.add_data(time=time, vel=rv, velerr=e_rv, instrument=instruments, t0=t0)

In [ ]:
tc_c_lower = 2459205.0 - 2457000  # BTJD
tc_c_upper = 2459225.0 - 2457000  # BTJD
tc_c = (tc_c_lower + tc_c_upper) / 2

params_2pl = {
    # Planet b (circular)
    "P_b": Parameter(1.5484, "d", fixed=False),
    "K_b": Parameter(10.0, "m/s", fixed=False),
    "secosw_b": Parameter(0.0, "", fixed=True),
    "sesinw_b": Parameter(0.0, "", fixed=True),
    "Tc_b": Parameter(tc_b, "d", fixed=False),
    # Planet c (eccentric)
    "P_c": Parameter(50.5, "d", fixed=False),
    "K_c": Parameter(10.0, "m/s", fixed=False),
    "secosw_c": Parameter(0.0, "", fixed=False),
    "sesinw_c": Parameter(0.0, "", fixed=False),
    "Tc_c": Parameter(tc_c, "d", fixed=False),
    # Trend (fixed)
    "gd": Parameter(0.0, "m/s/d", fixed=True),
    "gdd": Parameter(0.0, "m/s/d^2", fixed=True),
}
for inst in unique_instruments:
    mask = instruments == inst
    params_2pl[f"g_{inst}"] = Parameter(np.median(rv[mask]), "m/s", fixed=False)
    params_2pl[f"jit_{inst}"] = Parameter(0.0, "m/s", fixed=False)

priors_2pl = {
    # Planet b
    "P_b": Normal(1.5484, 0.000002),
    "K_b": Uniform(0.0, 50.0),
    "Tc_b": Normal(tc_b, 0.0007),
    # Planet c
    "P_c": Uniform(49.0, 52.0),
    "K_c": Uniform(0.0, 50.0),
    "e_c": HalfNormal(0.32),
    "w_c": Uniform(-np.pi, np.pi),
    "Tc_c": Uniform(tc_c_lower, tc_c_upper),
}
for inst in unique_instruments:
    priors_2pl[f"g_{inst}"] = Uniform(g_lower, g_upper)
    priors_2pl[f"jit_{inst}"] = Uniform(0, 30)

hyperparams_2pl = {
    "gp_amp": Parameter(1.0, "", fixed=False),
    "gp_lambda_e": Parameter(112, "d", fixed=False),
    "gp_lambda_p": Parameter(0.519, "", fixed=False),
    "gp_period": Parameter(19.343, "d", fixed=False),
}

hyperpriors_2pl = {
    "gp_amp": Uniform(0, rv_range),
    "gp_lambda_e": Uniform(6, 112 + (3 * 28)),
    "gp_lambda_p": Uniform(0.1, 0.519 + (4 * 0.091)),
    "gp_period": Uniform(19.343 - 2, 19.343 + 1),
}

fitter_2pl.params = params_2pl
fitter_2pl.priors = priors_2pl
fitter_2pl.hyperparams = hyperparams_2pl
fitter_2pl.hyperpriors = hyperpriors_2pl

print(f"Free parameters: {list(fitter_2pl.free_params_dict.keys())}")
print(f"Free hyperparameters: {list(fitter_2pl.free_hyperparams_names)}")
print(f"Total dimensions: {fitter_2pl.ndim}")

### Fit the 2-planet model

In [ ]:
map_2pl = fitter_2pl.find_map_estimate(method="Powell")
print(f"MAP log-probability: {-map_2pl.fun:.2f}")

In [ ]:
nwalkers_2pl = 32  # Needs to be at least 2x ndims, but you should use hundreds! 
init_2pl = fitter_2pl.generate_initial_walker_positions_random(nwalkers=nwalkers_2pl)

fitter_2pl.run_mcmc(
    initial_positions=init_2pl,
    nwalkers=nwalkers_2pl,
    max_steps=50_000,
    check_convergence=True,
    convergence_check_interval=1000,
    convergence_check_start=15_000,
    progress=True,
    multiprocessing=True,
)
print("MCMC completed")

In [ ]:
total_steps_2pl, nwalkers_2pl, ndim_2pl = fitter_2pl.sampler.iteration, fitter_2pl.sampler.nwalkers, fitter_2pl.sampler.ndim

# using the same samples_per_walker=1000 and thin_rate=10 as for 1pl model
discard_start_2pl = total_steps_2pl - (samples_per_walker * thin_rate)

print(f"nsteps: {total_steps_2pl}, nwalkers: {nwalkers_2pl}, ndim: {ndim_2pl}")
print(f"Discarding first {discard_start_2pl} steps, then thinning by {thin_rate} to get {samples_per_walker} samples per walker.")
print(f"Total samples after discarding and thinning: {(total_steps_2pl - discard_start_2pl) // thin_rate * nwalkers_2pl}")

In [ ]:
samples_2pl = fitter_2pl.get_samples_np(discard_start=discard_start_2pl, thin=thin_rate, flat=False)
lnprob_2pl = fitter_2pl.get_sampler_lnprob(discard_start=discard_start_2pl, thin=thin_rate, flat=False)
print(f"Samples shape: {samples_2pl.shape}  (nsteps, nwalkers, ndim)")
print(f"Lnprob shape:  {lnprob_2pl.shape}  (nsteps, nwalkers)")

In [ ]:
fitter_2pl.plot_corner(discard_start=discard_start_2pl, thin=thin_rate, plot_datapoints=True)

## Comparing the fits

Let's plot the RV model for both the 1-planet and 2-planet fits using their median values of the posterior parameters (obviously you should always check your posterior distributions rather than just taking these values blindly).

I've used custom xlims to zoom in on one clump of HARPS data to make it clearer - you can change these arguments to have a look at the full set of data.

In [ ]:
# Build median posterior parameter dicts for both models
samples_flat_1pl = fitter_1pl.get_samples_np(
    discard_start=discard_start_1pl, thin=thin_rate, flat=True  # flatten the (nsteps, nwalkers, ndim) array into (nsteps*nwalkers, ndim)
)
medians_1pl = np.percentile(samples_flat_1pl, 50, axis=0)
dict_1pl = fitter_1pl.build_params_dict(medians_1pl)

samples_flat_2pl = fitter_2pl.get_samples_np(
    discard_start=discard_start_2pl, thin=thin_rate, flat=True  # flatten the (nsteps, nwalkers, ndim) array into (nsteps*nwalkers, ndim)
)
medians_2pl = np.percentile(samples_flat_2pl, 50, axis=0)
dict_2pl = fitter_2pl.build_params_dict(medians_2pl)

In [ ]:
fitter_1pl.plot_custom_rv(dict_1pl, title="1-Planet Model (median posterior)", n_smooth=5000,
                          xlim=(2485, 2637), res_xlim=(2485, 2637),
                          ylim=(-23, 33))

In [ ]:
fitter_2pl.plot_custom_rv(dict_2pl, title="2-Planet Model (median posterior)", n_smooth=5000, 
                          xlim=(2485, 2637), res_xlim=(2485, 2637),
                          ylim=(-23, 33))

At first glance looking at the residuals, the two fits look quite similar, both models seem to do a reasonable job of modelling the data.

### Minimum planet masses

Let's investigate how the mass(es) differ between between the two models. We can compute $M_p \sin i$ for each model to see how the planet b mass estimate compares, and what the candidate planet c mass looks like. We use the value of $M_{\star}=0.630 \pm 0.018 M_{\odot}$ from Table 1 of Osborne et al. 2024.

In [ ]:
from ravest.model import calculate_mpsini

# Stellar mass from Osborne et al. (2024) Table 1
st_mass = 0.631  # M_sun
st_mass_err = 0.018  # M_sun

# --- 1-planet model: planet b only ---
post_1pl = fitter_1pl.get_mcmc_posterior_dict(discard_start=discard_start_1pl, thin=thin_rate)
n_samples_1pl = len(post_1pl["P_b"])
stellar_masses_1pl = np.random.normal(loc=st_mass, scale=st_mass_err, size=n_samples_1pl)

ecc_b_1pl = post_1pl["secosw_b"]**2 + post_1pl["sesinw_b"]**2  # get e from secosw and sesinw
mpsini_b_1pl = calculate_mpsini(stellar_masses_1pl, post_1pl["P_b"], post_1pl["K_b"], ecc_b_1pl, unit="M_earth")
p16, p50, p84 = np.percentile(mpsini_b_1pl, [16, 50, 84])
print("1-Planet Model:")
print(f"  Planet b: Mpsini = {p50:.2f} +{p84 - p50:.2f} -{p50 - p16:.2f} M_Earth")

# --- 2-planet model: planets b and c ---
post_2pl = fitter_2pl.get_mcmc_posterior_dict(discard_start=discard_start_2pl, thin=thin_rate)
n_samples_2pl = len(post_2pl["P_b"])
stellar_masses_2pl = np.random.normal(loc=st_mass, scale=st_mass_err, size=n_samples_2pl)

print("\n2-Planet Model:")
for planet in ["b", "c"]:
    ecc = post_2pl[f"secosw_{planet}"]**2 + post_2pl[f"sesinw_{planet}"]**2  # get e from secosw and sesinw
    mpsini = calculate_mpsini(stellar_masses_2pl, post_2pl[f"P_{planet}"], post_2pl[f"K_{planet}"], ecc, unit="M_earth")
    p16, p50, p84 = np.percentile(mpsini, [16, 50, 84])
    print(f"  Planet {planet}: Mpsini = {p50:.2f} +{p84 - p50:.2f} -{p50 - p16:.2f} M_Earth")

So while at first glance we have two quite similar fits, we can see that's two very different results! With the two-planet model, not only do we now have a second planet, but planet b decreases in mass by $\sim10$% - that could have a big impact on analyses downstream that use exoplanet masses (e.g. density, bulk compositions).

This motivates using the Learned Harmonic Mean Estimator (LHME) via the `harmonic` package to compare our models and see which one is preferred.

## Bayesian Evidence Estimation with harmonic

Now for the main event. We use `harmonic` to estimate the Bayesian evidence $\ln \mathcal{Z}$ for each model from the MCMC posterior samples. The workflow is:

1. Package the MCMC samples into harmonic's `Chains` format
2. Split into training (25%) and inference (75%) sets
3. Train a normalizing flow (`RQSplineModel`) on the training samples to learn the posterior distribution
4. Estimate the evidence from the inference samples

One important detail: ravest's `get_samples_np(flat=False)` returns arrays with shape `(nsteps, nwalkers, ndim)`, but harmonic's `add_chains_3d` expects `(nchains, nsamples, ndim)`. We need to transpose axes 0 and 1.

`harmonic` provides a few diagnostics to check the reliability of the evidence estimate. First, the `check_basic_diagnostic` command built in to Harmonic. Secondly, we check the **kurtosis** $\kappa$ which should be close to 3 (i.e. roughly Gaussian) -- anything far away from that indicates an unreliable estimate due to heavy tails. Thirdly, the **nu^2/sigma^2 ratio** ($\nu^2/\sigma^2$) compares the estimated variance of the variance to its theoretical target; values close to the target indicate sufficient effective samples. More information on these in §3.2 of [McEwen et al. (2021)](https://doi.org/10.48550/arXiv.2111.12720).

The main caveat is that all of the posteriors need to be unimodal, constrained and well-behaved: because the distribution `harmonic` makes needs to be constrained *within* the tails of the posterior distributions (i.e. within your samples). For full details on harmonic, see the [documentation](https://astro-informatics.github.io/harmonic/) and [GitHub repository](https://github.com/astro-informatics/harmonic).

### 1-Planet Model Evidence

In [ ]:
# Transpose from (nsteps, nwalkers, ndim) to (nchains, nsamples, ndim)
samples_1pl_hm = np.ascontiguousarray(samples_1pl.transpose(1, 0, 2))
lnprob_1pl_hm = np.ascontiguousarray(lnprob_1pl.T)

print(f"Samples shape for harmonic: {samples_1pl_hm.shape}  (nchains, nsamples, ndim)")
print(f"Lnprob shape for harmonic:  {lnprob_1pl_hm.shape}  (nchains, nsamples)")

chains_1pl = hm.Chains(ndim=fitter_1pl.ndim)
chains_1pl.add_chains_3d(samples=samples_1pl_hm, ln_posterior=lnprob_1pl_hm)

In [ ]:
chains_train_1pl, chains_test_1pl = hm.utils.split_data(
    chains_1pl, training_proportion=0.25
)

model_1pl = hm.model.RQSplineModel(
    fitter_1pl.ndim,
    n_layers=3,
    n_bins=8,
    hidden_size=[32, 32],
    spline_range=(-10.0, 10.0),
    standardize=True,
    temperature=0.8,
)

model_1pl.fit(chains_train_1pl.samples, epochs=100, batch_size=128, verbose=True)

In [ ]:
ev_1pl = hm.Evidence(nchains=chains_test_1pl.nchains, model=model_1pl)
ev_1pl.add_chains(chains_test_1pl)

ln_Z_1pl, ln_Z_1pl_std = ev_1pl.compute_ln_evidence()
check_1pl = ev_1pl.check_basic_diagnostic()

kurtosis_1pl = ev_1pl.kurtosis
n_eff_1pl = ev_1pl.n_eff
nu2_sigma2_1pl = np.exp(0.5 * ev_1pl.ln_evidence_inv_var_var - ev_1pl.ln_evidence_inv_var)
target_nu2_sigma2_1pl = np.sqrt(2.0 / (n_eff_1pl - 1))

print(f"1-Planet Model: ln(Z) = {ln_Z_1pl:.3f} +/- {ln_Z_1pl_std:.3f}")
print(f"Basic diagnostic:  {'PASS' if check_1pl else 'FAIL'}")
print(f"Kurtosis:          {kurtosis_1pl:.3f}  (aim for ~3)")
print(f"nu^2/sigma^2:      {nu2_sigma2_1pl:.4f}  (target: {target_nu2_sigma2_1pl:.4f})")
print(f"n_eff:             {n_eff_1pl:.1f}")

### 2-Planet Model Evidence

In [ ]:
samples_2pl_hm = np.ascontiguousarray(samples_2pl.transpose(1, 0, 2))
lnprob_2pl_hm = np.ascontiguousarray(lnprob_2pl.T)

print(f"Samples shape for harmonic: {samples_2pl_hm.shape}  (nchains, nsamples, ndim)")
print(f"Lnprob shape for harmonic:  {lnprob_2pl_hm.shape}  (nchains, nsamples)")

chains_2pl = hm.Chains(ndim=fitter_2pl.ndim)
chains_2pl.add_chains_3d(samples=samples_2pl_hm, ln_posterior=lnprob_2pl_hm)

In [ ]:
chains_train_2pl, chains_test_2pl = hm.utils.split_data(
    chains_2pl, training_proportion=0.25
)

model_2pl = hm.model.RQSplineModel(
    fitter_2pl.ndim,
    n_layers=3,
    n_bins=8,
    hidden_size=[32, 32],
    spline_range=(-10.0, 10.0),
    standardize=True,
    temperature=0.8,
)

model_2pl.fit(chains_train_2pl.samples, epochs=100, batch_size=128, verbose=True)

In [ ]:
ev_2pl = hm.Evidence(nchains=chains_test_2pl.nchains, model=model_2pl)
ev_2pl.add_chains(chains_test_2pl)

ln_Z_2pl, ln_Z_2pl_std = ev_2pl.compute_ln_evidence()
check_2pl = ev_2pl.check_basic_diagnostic()

kurtosis_2pl = ev_2pl.kurtosis
n_eff_2pl = ev_2pl.n_eff
nu2_sigma2_2pl = np.exp(0.5 * ev_2pl.ln_evidence_inv_var_var - ev_2pl.ln_evidence_inv_var)
target_nu2_sigma2_2pl = np.sqrt(2.0 / (n_eff_2pl - 1))

print(f"2-Planet Model: ln(Z) = {ln_Z_2pl:.3f} +/- {ln_Z_2pl_std:.3f}")
print(f"Basic diagnostic:  {'PASS' if check_2pl else 'FAIL'}")
print(f"Kurtosis:          {kurtosis_2pl:.3f}  (aim for ~3)")
print(f"nu^2/sigma^2:      {nu2_sigma2_2pl:.4f}  (target: {target_nu2_sigma2_2pl:.4f})")
print(f"n_eff:             {n_eff_2pl:.1f}")

## Model Comparison

In [ ]:
delta_ln_Z = ln_Z_2pl - ln_Z_1pl

print(f"{'Model':<20} {'ln(Z)':>12} {'std':>8}")
print("-" * 42)
print(f"{'1-Planet':<20} {ln_Z_1pl:>12.3f} {ln_Z_1pl_std:>8.3f}")
print(f"{'2-Planet':<20} {ln_Z_2pl:>12.3f} {ln_Z_2pl_std:>8.3f}")
print(f"\nDelta ln(Z) = ln(Z_2pl) - ln(Z_1pl) = {delta_ln_Z:.3f}")

We can interpret the (log-)Bayes factor $\Delta\ln{\mathcal{Z}}$ using the scale given in [Trotta (2008)](https://doi.org/10.1080/00107510802066753): inconclusive for $<1.0$, weak evidence up to $2.5$, moderate evidence up to $5.0$, and strong evidence above that.

We find $\ln \mathcal{Z}_{\text{1pl}} = -385.534$ and $\ln \mathcal{Z}_{\text{2pl}} = -366.466$, giving $\Delta \ln \mathcal{Z_{21}} = \ln \mathcal{Z_{2}} - \ln \mathcal{Z_{1}} = 19.1$ -- **very strong evidence in favour of the 2-planet model!** It goes to show that if you aren't careful and test alternative models, you could end up with a result that looks good at first glance (the one-planet model) when another solution may be preferable, with big implications for the masses (and everything that uses masses downstream.)

Note that unlike AIC or BIC, the Bayesian evidence naturally penalises model complexity through the prior volume. AIC and BIC penalise models only based on the number of free parameters and the likelihood; the priors aren't involved. A more complex model must fit the data *sufficiently* better to justify the additional parameters (incorporating the increased prior volume) -- the Bayesian Occam's razor is built in. Obviously though, this means you should have normalised, justified, well-behaved sensible priors for all parameters and hyperparameters, or it may not be a fair comparison!

### Comparison with information criteria

As a quick check, we can also compare the models using classical information criteria.

In [ ]:
chi2_1pl = fitter_1pl.calculate_chi2(dict_1pl)
aicc_1pl = fitter_1pl.calculate_aicc(dict_1pl)
bic_1pl = fitter_1pl.calculate_bic(dict_1pl)

chi2_2pl = fitter_2pl.calculate_chi2(dict_2pl)
aicc_2pl = fitter_2pl.calculate_aicc(dict_2pl)
bic_2pl = fitter_2pl.calculate_bic(dict_2pl)

print(f"{'Metric':<20} {'1-Planet':>12} {'2-Planet':>12}")
print("-" * 46)
print(f"{'Free parameters':<20} {fitter_1pl.ndim:>12} {fitter_2pl.ndim:>12}")
print(f"{'Chi2':<20} {chi2_1pl:>12.2f} {chi2_2pl:>12.2f}")
print(f"{'AICc':<20} {aicc_1pl:>12.2f} {aicc_2pl:>12.2f} (lower is better)")
print(f"{'BIC':<20} {bic_1pl:>12.2f} {bic_2pl:>12.2f} (lower is better)")
print(f"{'ln(Z) [harmonic]':<20} {ln_Z_1pl:>12.3f} {ln_Z_2pl:>12.3f} (higher is better)")

## Summary

We used `harmonic` to estimate the Bayesian evidence for 1-planet and 2-planet models of TOI-544. The key steps were:

1. Fit both models with ravest's `GPFitter` to obtain posterior MCMC samples
2. Transpose the samples from ravest's `(nsteps, nwalkers, ndim)` to harmonic's `(nchains, nsamples, ndim)` format
3. Train an `RQSplineModel` normalizing flow on a subset of samples
4. Estimate the evidence from the remaining samples

The 2-planet model is strongly preferred, consistent with both the Bayesian evidence and classical information criteria. This shows the importance of testing your models to ensure you have the most appropriate one. You could consider testing not only the number of planets, but maybe circular vs eccentric orbits, the choices of priors and hyperiors, whether to use a GP (and different kernels), whether to fit a long-term linear or quadratic trend. Do bear in mind that you need to be comparing the same *data* in your models each time though.

**Further reading:**
- [harmonic documentation](https://astro-informatics.github.io/harmonic/)
- [McEwen et al. (2021)](https://doi.org/10.48550/arXiv.2111.12720) --- the learned harmonic mean estimator
- [Trotta (2008)](https://doi.org/10.1080/00107510802066753) --- Bayesian model comparison review